In [ ]:

import gc
import math
import os
import random
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from imblearn.over_sampling import SMOTE
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.utils.class_weight import compute_class_weight


SEEDS = [11, 22, 33, 44, 55]
PARTITION_SEED = 2026

LABEL_COLUMN = "label"
BINARY_COLUMN = "binary_label"
SOURCE_COLUMN = "source"
ORIGINAL_LABEL_COLUMN = "original_label"

MAX_TRAIN_ROWS = None
MAX_TEST_ROWS = None

EPOCHS = {
    "alzughabi_pso_dnn": 100,
    "wang_cnn": 50,
    "wang_rnn": 50,
    "wang_lstm": 50,
    "anwar_lightweight_1dcnn": 20,
}

BATCH_SIZES = {
    "alzughabi_pso_dnn": 2048,
    "wang_cnn": 2048,
    "wang_rnn": 2048,
    "wang_lstm": 2048,
    "anwar_lightweight_1dcnn": 2048,
}

MODEL_KEYS = [
    "alzughabi_pso_dnn",
    "wang_cnn",
    "wang_rnn",
    "wang_lstm",
    "anwar_lightweight_1dcnn",
    "ensemble_feature_rf",
]

RESULTS_DIR = Path(
    "/content/drive/MyDrive/Array_submission/Results/"
    "CSE_CIC_IDS2018/Experiment4_model_specific_test_ratios"
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Results directory:", RESULTS_DIR)

TensorFlow: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Results directory: /content/drive/MyDrive/Array_submission/Results/CSE_CIC_IDS2018/Experiment4_model_specific_test_ratios


In [ ]:

def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def binary_y(frame):
    return frame[BINARY_COLUMN].to_numpy(dtype=np.int8)


def optional_sample(frame, maximum_rows, seed):
    if maximum_rows is None or len(frame) <= maximum_rows:
        return frame.reset_index(drop=True).copy()

    sampled, _ = train_test_split(
        frame,
        train_size=maximum_rows,
        random_state=seed,
        shuffle=True,
        stratify=frame[BINARY_COLUMN],
    )
    return sampled.reset_index(drop=True)


def prepare_test(frame, seed):
    return optional_sample(
        frame,
        MAX_TEST_ROWS,
        seed,
    )


def standard_callbacks(
    monitor="val_loss",
    mode="min",
    patience=5,
):
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor=monitor,
            mode=mode,
            patience=patience,
            restore_best_weights=True,
            min_delta=1e-4,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor=monitor,
            mode=mode,
            factor=0.5,
            patience=max(2, patience // 2),
            min_lr=1e-7,
            verbose=1,
        ),
    ]

In [ ]:

NATIVE_FILE_PATHS = {
    "2018-02-14": (
        "/content/drive/MyDrive/Colab Notebooks/DATASETS/"
        "CIC-IDS-2018/Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv"
    ),
    "2018-02-15": (
        "/content/drive/MyDrive/Colab Notebooks/DATASETS/"
        "CIC-IDS-2018/Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv"
    ),
    "2018-02-16": (
        "/content/drive/MyDrive/Colab Notebooks/DATASETS/"
        "CIC-IDS-2018/Friday-16-02-2018_TrafficForML_CICFlowMeter.csv"
    ),
    "2018-02-20": (
        "/content/drive/MyDrive/Colab Notebooks/DATASETS/"
        "CIC-IDS-2018/Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv"
    ),
    "2018-02-21": (
        "/content/drive/MyDrive/Colab Notebooks/DATASETS/"
        "CIC-IDS-2018/Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv"
    ),
    "2018-02-22": (
        "/content/drive/MyDrive/Colab Notebooks/DATASETS/"
        "CIC-IDS-2018/Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv"
    ),
    "2018-02-23": (
        "/content/drive/MyDrive/Colab Notebooks/DATASETS/"
        "CIC-IDS-2018/Friday-23-02-2018_TrafficForML_CICFlowMeter.csv"
    ),
    "2018-02-28": (
        "/content/drive/MyDrive/Colab Notebooks/DATASETS/"
        "CIC-IDS-2018/Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv"
    ),
    "2018-03-01": (
        "/content/drive/MyDrive/Colab Notebooks/DATASETS/"
        "CIC-IDS-2018/Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv"
    ),
    "2018-03-02": (
        "/content/drive/MyDrive/Colab Notebooks/DATASETS/"
        "CIC-IDS-2018/Friday-02-03-2018_TrafficForML_CICFlowMeter.csv"
    ),
}

native_parts = []

for capture_date, path in NATIVE_FILE_PATHS.items():
    part = pd.read_csv(path)
    part["capture_date"] = capture_date
    native_parts.append(part)

native_raw = pd.concat(
    native_parts,
    ignore_index=True,
)

del native_parts
gc.collect()

print("Native raw shape:", native_raw.shape)
print(native_raw["capture_date"].value_counts().sort_index())

/tmp/ipykernel_20431/1916146161.py:49: DtypeWarning: Columns (0,1,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78) have mixed types. Specify dtype option on import or set low_memory=False.
  part = pd.read_csv(path)
/tmp/ipykernel_20431/1916146161.py:49: DtypeWarning: Columns (0,1,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78) have mixed types. Specify dtype option on import or set low_memory=False.
  part = pd.read_csv(path)
/tmp/ipykernel_20431/1916146161.py:49: DtypeWarning: Columns (0,1,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,5

Native raw shape: (16233002, 85)
capture_date
2018-02-14    1048575
2018-02-15    1048575
2018-02-16    1048575
2018-02-20    7948748
2018-02-21    1048575
2018-02-22    1048575
2018-02-23    1048575
2018-02-28     613104
2018-03-01     331125
2018-03-02    1048575
Name: count, dtype: int64


In [ ]:

darpa_raw = pd.read_csv(
    "/content/drive/MyDrive/Array_submission/Dataset/"
    "CIC-features/Darpa/Darpa_Pcap_CIC_features.csv"
)
unsw_raw = pd.read_csv(
    "/content/drive/MyDrive/Array_submission/Dataset/"
    "CIC-features/UNSW-NB15/UNSW_Pcap_CIC_features.csv"
)
cic17_raw = pd.read_csv(
    "/content/drive/MyDrive/Array_submission/Dataset/"
    "CIC-features/CIC2017/CIC17_Pcap_CIC_features.csv"
)
cic18_raw = pd.read_csv(
    "/content/drive/MyDrive/Array_submission/Dataset/"
    "CIC-features/CIC2018/CIC18_Pcap_CIC_features.csv"
)

print("DARPA:", darpa_raw.shape)
print("UNSW-NB15:", unsw_raw.shape)
print("CIC-IDS2017:", cic17_raw.shape)
print("CSE-CIC-IDS2018:", cic18_raw.shape)

/tmp/ipykernel_20431/3412680032.py:15: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77) have mixed types. Specify dtype option on import or set low_memory=False.
  cic18_raw = pd.read_csv(


DARPA: (2439149, 95)
UNSW-NB15: (313636, 84)
CIC-IDS2017: (1233126, 94)
CSE-CIC-IDS2018: (1661679, 79)


In [10]:
CSE2018_FEATURE_COLUMNS = [
    "Dst Port",
    "Flow Duration",
    "Tot Fwd Pkts",
    "Tot Bwd Pkts",
    "TotLen Fwd Pkts",
    "TotLen Bwd Pkts",
    "Fwd Pkt Len Max",
    "Fwd Pkt Len Min",
    "Fwd Pkt Len Mean",
    "Fwd Pkt Len Std",
    "Bwd Pkt Len Max",
    "Bwd Pkt Len Min",
    "Bwd Pkt Len Mean",
    "Bwd Pkt Len Std",
    "Flow Byts/s",
    "Flow Pkts/s",
    "Flow IAT Mean",
    "Flow IAT Std",
    "Flow IAT Max",
    "Flow IAT Min",
    "Fwd IAT Tot",
    "Fwd IAT Mean",
    "Fwd IAT Std",
    "Fwd IAT Max",
    "Fwd IAT Min",
    "Bwd IAT Tot",
    "Bwd IAT Mean",
    "Bwd IAT Std",
    "Bwd IAT Max",
    "Bwd IAT Min",
    "Fwd PSH Flags",
    "Bwd PSH Flags",
    "Fwd URG Flags",
    "Bwd URG Flags",
    "Fwd Header Len",
    "Bwd Header Len",
    "Fwd Pkts/s",
    "Bwd Pkts/s",
    "Pkt Len Min",
    "Pkt Len Max",
    "Pkt Len Mean",
    "Pkt Len Std",
    "Pkt Len Var",
    "FIN Flag Cnt",
    "SYN Flag Cnt",
    "RST Flag Cnt",
    "PSH Flag Cnt",
    "ACK Flag Cnt",
    "URG Flag Cnt",
    "CWR Flag Count",
    "ECE Flag Cnt",
    "Down/Up Ratio",
    "Pkt Size Avg",
    "Fwd Seg Size Avg",
    "Bwd Seg Size Avg",
    "Fwd Byts/b Avg",
    "Fwd Pkts/b Avg",
    "Fwd Blk Rate Avg",
    "Bwd Byts/b Avg",
    "Bwd Pkts/b Avg",
    "Bwd Blk Rate Avg",
    "Subflow Fwd Pkts",
    "Subflow Fwd Byts",
    "Subflow Bwd Pkts",
    "Subflow Bwd Byts",
    "Init Fwd Win Byts",
    "Init Bwd Win Byts",
    "Fwd Act Data Pkts",
    "Fwd Seg Size Min",
    "Active Mean",
    "Active Std",
    "Active Max",
    "Active Min",
    "Idle Mean",
    "Idle Std",
    "Idle Max",
    "Idle Min",
]

CIC_TO_CSE2018_ALIASES = {
    "Destination Port": "Dst Port",
    "dst_port": "Dst Port",
    "protocol": "Protocol",
    "flow_duration": "Flow Duration",
    "Total Fwd Packets": "Tot Fwd Pkts",
    "Total Fwd Packet": "Tot Fwd Pkts",
    "tot_fwd_pkts": "Tot Fwd Pkts",
    "Total Backward Packets": "Tot Bwd Pkts",
    "Total Bwd packets": "Tot Bwd Pkts",
    "tot_bwd_pkts": "Tot Bwd Pkts",
    "Total Length of Fwd Packets": "TotLen Fwd Pkts",
    "Total Length of Fwd Packet": "TotLen Fwd Pkts",
    "totlen_fwd_pkts": "TotLen Fwd Pkts",
    "Total Length of Bwd Packets": "TotLen Bwd Pkts",
    "Total Length of Bwd Packet": "TotLen Bwd Pkts",
    "totlen_bwd_pkts": "TotLen Bwd Pkts",
    "Fwd Packet Length Max": "Fwd Pkt Len Max",
    "fwd_pkt_len_max": "Fwd Pkt Len Max",
    "Fwd Packet Length Min": "Fwd Pkt Len Min",
    "fwd_pkt_len_min": "Fwd Pkt Len Min",
    "Fwd Packet Length Mean": "Fwd Pkt Len Mean",
    "fwd_pkt_len_mean": "Fwd Pkt Len Mean",
    "Fwd Packet Length Std": "Fwd Pkt Len Std",
    "fwd_pkt_len_std": "Fwd Pkt Len Std",
    "Bwd Packet Length Max": "Bwd Pkt Len Max",
    "bwd_pkt_len_max": "Bwd Pkt Len Max",
    "Bwd Packet Length Min": "Bwd Pkt Len Min",
    "bwd_pkt_len_min": "Bwd Pkt Len Min",
    "Bwd Packet Length Mean": "Bwd Pkt Len Mean",
    "bwd_pkt_len_mean": "Bwd Pkt Len Mean",
    "Bwd Packet Length Std": "Bwd Pkt Len Std",
    "bwd_pkt_len_std": "Bwd Pkt Len Std",
    "Flow Bytes/s": "Flow Byts/s",
    "flow_byts_s": "Flow Byts/s",
    "Flow Packets/s": "Flow Pkts/s",
    "flow_pkts_s": "Flow Pkts/s",
    "flow_iat_mean": "Flow IAT Mean",
    "flow_iat_std": "Flow IAT Std",
    "flow_iat_max": "Flow IAT Max",
    "flow_iat_min": "Flow IAT Min",
    "Fwd IAT Total": "Fwd IAT Tot",
    "fwd_iat_tot": "Fwd IAT Tot",
    "fwd_iat_mean": "Fwd IAT Mean",
    "fwd_iat_std": "Fwd IAT Std",
    "fwd_iat_max": "Fwd IAT Max",
    "fwd_iat_min": "Fwd IAT Min",
    "Bwd IAT Total": "Bwd IAT Tot",
    "bwd_iat_tot": "Bwd IAT Tot",
    "bwd_iat_mean": "Bwd IAT Mean",
    "bwd_iat_std": "Bwd IAT Std",
    "bwd_iat_max": "Bwd IAT Max",
    "bwd_iat_min": "Bwd IAT Min",
    "fwd_psh_flags": "Fwd PSH Flags",
    "bwd_psh_flags": "Bwd PSH Flags",
    "fwd_urg_flags": "Fwd URG Flags",
    "bwd_urg_flags": "Bwd URG Flags",
    "Fwd Header Length": "Fwd Header Len",
    "fwd_header_len": "Fwd Header Len",
    "Bwd Header Length": "Bwd Header Len",
    "bwd_header_len": "Bwd Header Len",
    "Fwd Packets/s": "Fwd Pkts/s",
    "fwd_pkts_s": "Fwd Pkts/s",
    "Bwd Packets/s": "Bwd Pkts/s",
    "bwd_pkts_s": "Bwd Pkts/s",
    "Min Packet Length": "Pkt Len Min",
    "Packet Length Min": "Pkt Len Min",
    "pkt_len_min": "Pkt Len Min",
    "Max Packet Length": "Pkt Len Max",
    "Packet Length Max": "Pkt Len Max",
    "pkt_len_max": "Pkt Len Max",
    "Packet Length Mean": "Pkt Len Mean",
    "pkt_len_mean": "Pkt Len Mean",
    "Packet Length Std": "Pkt Len Std",
    "pkt_len_std": "Pkt Len Std",
    "Packet Length Variance": "Pkt Len Var",
    "pkt_len_var": "Pkt Len Var",
    "FIN Flag Count": "FIN Flag Cnt",
    "fin_flag_cnt": "FIN Flag Cnt",
    "SYN Flag Count": "SYN Flag Cnt",
    "syn_flag_cnt": "SYN Flag Cnt",
    "RST Flag Count": "RST Flag Cnt",
    "rst_flag_cnt": "RST Flag Cnt",
    "PSH Flag Count": "PSH Flag Cnt",
    "psh_flag_cnt": "PSH Flag Cnt",
    "ACK Flag Count": "ACK Flag Cnt",
    "ack_flag_cnt": "ACK Flag Cnt",
    "URG Flag Count": "URG Flag Cnt",
    "urg_flag_cnt": "URG Flag Cnt",
    "CWE Flag Count": "CWR Flag Count",
    "cwr_flag_count": "CWR Flag Count",
    "ECE Flag Count": "ECE Flag Cnt",
    "ece_flag_cnt": "ECE Flag Cnt",
    "down_up_ratio": "Down/Up Ratio",
    "Average Packet Size": "Pkt Size Avg",
    "pkt_size_avg": "Pkt Size Avg",
    "Avg Fwd Segment Size": "Fwd Seg Size Avg",
    "fwd_seg_size_avg": "Fwd Seg Size Avg",
    "Avg Bwd Segment Size": "Bwd Seg Size Avg",
    "bwd_seg_size_avg": "Bwd Seg Size Avg",
    "Fwd Avg Bytes/Bulk": "Fwd Byts/b Avg",
    "Fwd Bytes/Bulk Avg": "Fwd Byts/b Avg",
    "fwd_byts_b_avg": "Fwd Byts/b Avg",
    "Fwd Avg Packets/Bulk": "Fwd Pkts/b Avg",
    "Fwd Packet/Bulk Avg": "Fwd Pkts/b Avg",
    "fwd_pkts_b_avg": "Fwd Pkts/b Avg",
    "Fwd Avg Bulk Rate": "Fwd Blk Rate Avg",
    "Fwd Bulk Rate Avg": "Fwd Blk Rate Avg",
    "fwd_blk_rate_avg": "Fwd Blk Rate Avg",
    "Bwd Avg Bytes/Bulk": "Bwd Byts/b Avg",
    "Bwd Bytes/Bulk Avg": "Bwd Byts/b Avg",
    "bwd_byts_b_avg": "Bwd Byts/b Avg",
    "Bwd Avg Packets/Bulk": "Bwd Pkts/b Avg",
    "Bwd Packet/Bulk Avg": "Bwd Pkts/b Avg",
    "bwd_pkts_b_avg": "Bwd Pkts/b Avg",
    "Bwd Avg Bulk Rate": "Bwd Blk Rate Avg",
    "Bwd Bulk Rate Avg": "Bwd Blk Rate Avg",
    "bwd_blk_rate_avg": "Bwd Blk Rate Avg",
    "Subflow Fwd Packets": "Subflow Fwd Pkts",
    "subflow_fwd_pkts": "Subflow Fwd Pkts",
    "Subflow Fwd Bytes": "Subflow Fwd Byts",
    "subflow_fwd_byts": "Subflow Fwd Byts",
    "Subflow Bwd Packets": "Subflow Bwd Pkts",
    "subflow_bwd_pkts": "Subflow Bwd Pkts",
    "Subflow Bwd Bytes": "Subflow Bwd Byts",
    "subflow_bwd_byts": "Subflow Bwd Byts",
    "Init_Win_bytes_forward": "Init Fwd Win Byts",
    "FWD Init Win Bytes": "Init Fwd Win Byts",
    "init_fwd_win_byts": "Init Fwd Win Byts",
    "Init_Win_bytes_backward": "Init Bwd Win Byts",
    "Bwd Init Win Bytes": "Init Bwd Win Byts",
    "init_bwd_win_byts": "Init Bwd Win Byts",
    "act_data_pkt_fwd": "Fwd Act Data Pkts",
    "Fwd Act Data Pkts": "Fwd Act Data Pkts",
    "fwd_act_data_pkts": "Fwd Act Data Pkts",
    "min_seg_size_forward": "Fwd Seg Size Min",
    "fwd_seg_size_min": "Fwd Seg Size Min",
    "active_mean": "Active Mean",
    "active_std": "Active Std",
    "active_max": "Active Max",
    "active_min": "Active Min",
    "idle_mean": "Idle Mean",
    "idle_std": "Idle Std",
    "idle_max": "Idle Max",
    "idle_min": "Idle Min",
    "Fwd Segment Size Avg": "Fwd Seg Size Avg",
  "Bwd Segment Size Avg": "Bwd Seg Size Avg"

}

In [ ]:

BENIGN_TOKENS = {
    "benign",
    "normal",
    "normal traffic",
    "non attack",
    "non-attack",
    "0",
}


def normalize_text(value):
    if pd.isna(value):
        return ""

    text = str(value)
    text = text.replace("\ufeff", "")
    text = text.replace("�", "-")
    text = text.strip().strip("'\"").casefold()
    text = text.rstrip(".")
    text = re.sub(r"[_/]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_label(value):
    text = normalize_text(value)

    if text in BENIGN_TOKENS or text.startswith("benign"):
        return "benign"

    if (
        "ddos" in text
        or "distributed denial" in text
        or "hoic" in text
        or "loic" in text
    ):
        return "ddos"

    if (
        text == "dos"
        or "denial of service" in text
        or "slowloris" in text
        or "slowhttptest" in text
        or "goldeneye" in text
        or "hulk" in text
    ):
        return "dos"

    if (
        "portscan" in text
        or "port scan" in text
        or "reconnaissance" in text
        or text in {"recon", "probe"}
        or "nmap" in text
        or "ipsweep" in text
        or "portsweep" in text
        or "satan" in text
    ):
        return "reconnaissance"

    if (
        "brute force" in text
        or "bruteforce" in text
        or "patator" in text
        or "password" in text
    ):
        return "brute_force"

    if (
        "web attack" in text
        or "sql injection" in text
        or "xss" in text
        or "cross site" in text
    ):
        return "web_attack"

    if "bot" in text:
        return "botnet"

    if "infiltration" in text or "infilteration" in text:
        return "infiltration"

    if "heartbleed" in text:
        return "heartbleed"

    if "exploit" in text:
        return "exploits"

    if "fuzz" in text:
        return "fuzzers"

    if text == "generic" or "generic attack" in text:
        return "generic"

    if "backdoor" in text or "back door" in text:
        return "backdoor"

    if "shellcode" in text:
        return "shellcode"

    if "worm" in text:
        return "worms"

    if text == "r2l" or "remote to local" in text:
        return "r2l"

    if text == "u2r" or "user to root" in text:
        return "u2r"

    if text == "analysis":
        return "analysis"

    return "other_attack"


def normalize_exact_subtype(value):
    text = normalize_text(value)
    text = re.sub(
        r"(?:\s*[-]\s*|\s+)attempted\s*$",
        "",
        text,
    )
    return re.sub(r"\s+", " ", text).strip()

In [ ]:

def normalize_columns(frame):
    result = frame.copy()
    result.columns = (
        result.columns
        .astype(str)
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
        .str.strip("'\"")
    )
    return result


def coalesce_duplicate_columns(frame):
    if not frame.columns.duplicated().any():
        return frame

    output = pd.DataFrame(index=frame.index)

    for column in pd.unique(frame.columns):
        matching = frame.loc[:, frame.columns == column]

        if matching.shape[1] == 1:
            output[column] = matching.iloc[:, 0]
        else:
            output[column] = matching.bfill(axis=1).iloc[:, 0]

    return output


def find_label_column(frame, candidates):
    lookup = {
        normalize_text(column): column
        for column in frame.columns
    }

    for candidate in candidates:
        key = normalize_text(candidate)
        if key in lookup:
            return lookup[key]

    raise KeyError(
        f"No label column found. Candidates={candidates}; "
        f"available={frame.columns.tolist()}"
    )


def prepare_cse_schema_frame(
    raw_frame,
    source,
    label_candidates,
    capture_date_column=None,
):
    frame = normalize_columns(raw_frame)

    frame = frame.rename(
        columns={
            column: CIC_TO_CSE2018_ALIASES[column]
            for column in frame.columns
            if column in CIC_TO_CSE2018_ALIASES
        }
    )

    frame = coalesce_duplicate_columns(frame)

    label_column = find_label_column(
        frame,
        label_candidates,
    )

    metadata = pd.DataFrame(index=frame.index)
    metadata[ORIGINAL_LABEL_COLUMN] = (
        frame[label_column]
        .astype("string")
        .str.strip()
    )
    metadata[SOURCE_COLUMN] = source

    if (
        capture_date_column is not None
        and capture_date_column in frame.columns
    ):
        metadata["capture_date"] = (
            frame[capture_date_column]
            .astype("string")
        )

    missing_features = [
        column
        for column in CSE2018_FEATURE_COLUMNS
        if column not in frame.columns
    ]

    if missing_features:
        raise KeyError(
            f"{source} is missing CSE-CIC-IDS2018 features: "
            f"{missing_features}"
        )

    features = frame[
        CSE2018_FEATURE_COLUMNS
    ].copy()

    for column in CSE2018_FEATURE_COLUMNS:
        features[column] = pd.to_numeric(
            features[column],
            errors="coerce",
        )

    features = features.replace(
        [np.inf, -np.inf],
        np.nan,
    )

    result = pd.concat(
        [
            features.reset_index(drop=True),
            metadata.reset_index(drop=True),
        ],
        axis=1,
    )

    result[LABEL_COLUMN] = result[
        ORIGINAL_LABEL_COLUMN
    ].map(normalize_label)

    result[BINARY_COLUMN] = (
        result[LABEL_COLUMN].ne("benign")
    ).astype(np.int8)

    return result

In [ ]:

native_df = prepare_cse_schema_frame(
    native_raw,
    source="CSE-CIC-IDS2018-native",
    label_candidates=["Label", "label"],
    capture_date_column="capture_date",
)

darpa_pcap = prepare_cse_schema_frame(
    darpa_raw,
    source="DARPA",
    label_candidates=[
        "Attack_category",
        "attack_category",
        "Label",
        "label",
    ],
)

unsw_pcap = prepare_cse_schema_frame(
    unsw_raw,
    source="UNSW-NB15",
    label_candidates=[
        "category",
        "attack_cat",
        "Label",
        "label",
    ],
)

cic17_pcap = prepare_cse_schema_frame(
    cic17_raw,
    source="CIC-IDS2017",
    label_candidates=["Label", "label"],
)

cic18_pcap = prepare_cse_schema_frame(
    cic18_raw,
    source="CSE-CIC-IDS2018",
    label_candidates=["Label", "label"],
)

df_master = pd.concat(
    [
        darpa_pcap,
        unsw_pcap,
        cic17_pcap,
        cic18_pcap,
    ],
    ignore_index=True,
)

print("Native shape:", native_df.shape)
print("External master shape:", df_master.shape)
print("Feature count:", len(CSE2018_FEATURE_COLUMNS))

Native shape: (16233002, 82)
External master shape: (5647590, 81)
Feature count: 77


In [ ]:

schema_audit = pd.DataFrame(
    [
        {
            "dataset": name,
            "rows": len(frame),
            "feature_count": len(CSE2018_FEATURE_COLUMNS),
            "missing_feature_cells": int(
                frame[CSE2018_FEATURE_COLUMNS]
                .isna()
                .sum()
                .sum()
            ),
            "benign_rows": int(
                (frame[BINARY_COLUMN] == 0).sum()
            ),
            "attack_rows": int(
                (frame[BINARY_COLUMN] == 1).sum()
            ),
        }
        for name, frame in {
            "CSE-CIC-IDS2018-native": native_df,
            "DARPA": darpa_pcap,
            "UNSW-NB15": unsw_pcap,
            "CIC-IDS2017": cic17_pcap,
            "CSE-CIC-IDS2018": cic18_pcap,
        }.items()
    ]
)

display(schema_audit)

label_audit = (
    pd.concat(
        [native_df, df_master],
        ignore_index=True,
    )
    .groupby(
        [SOURCE_COLUMN, LABEL_COLUMN],
        dropna=False,
    )
    .size()
    .rename("rows")
    .reset_index()
)

display(label_audit)

,dataset,rows,feature_count,missing_feature_cells,benign_rows,attack_rows
0,CSE-CIC-IDS2018-native,16233002,77,196063,13484708,2748294
1,DARPA,2439149,77,0,2264038,175111
2,UNSW-NB15,313636,77,0,299097,14539
3,CIC-IDS2017,1233126,77,0,1055350,177776
4,CSE-CIC-IDS2018,1661679,77,30933,1540277,121402


,source,label,rows
0,CIC-IDS2017,benign,1055350
1,CIC-IDS2017,dos,177765
2,CIC-IDS2017,web_attack,11
3,CSE-CIC-IDS2018,benign,1540277
4,CSE-CIC-IDS2018,dos,52498
5,CSE-CIC-IDS2018,infiltration,68871
6,CSE-CIC-IDS2018,other_attack,33
7,CSE-CIC-IDS2018-native,benign,13484708
8,CSE-CIC-IDS2018-native,botnet,286191
9,CSE-CIC-IDS2018-native,brute_force,381790


In [ ]:

def numeric_matrix(frame, feature_columns):
    matrix = frame.reindex(
        columns=feature_columns,
    ).copy()

    for column in feature_columns:
        matrix[column] = pd.to_numeric(
            matrix[column],
            errors="coerce",
        )

    return matrix.replace(
        [np.inf, -np.inf],
        np.nan,
    )


def fit_numeric_preprocessor(
    train_frame,
    feature_columns,
    scaler_kind,
    drop_constant=True,
):
    matrix = numeric_matrix(
        train_frame,
        feature_columns,
    )

    medians = matrix.median(axis=0).fillna(0.0)
    matrix = matrix.fillna(medians)

    active_features = list(feature_columns)

    if drop_constant:
        active_features = [
            column
            for column in feature_columns
            if matrix[column].nunique(
                dropna=False
            ) > 1
        ]

    if not active_features:
        raise ValueError(
            "No non-constant training features remain."
        )

    matrix = matrix[active_features]

    if scaler_kind == "standard":
        scaler = StandardScaler()
    elif scaler_kind == "minmax":
        scaler = MinMaxScaler(
            feature_range=(0, 1)
        )
    elif scaler_kind == "none":
        scaler = None
    else:
        raise ValueError(
            f"Unknown scaler kind: {scaler_kind}"
        )

    transformed = (
        scaler.fit_transform(matrix)
        if scaler is not None
        else matrix.to_numpy()
    )

    return (
        {
            "feature_columns": active_features,
            "medians": medians,
            "scaler": scaler,
            "scaler_kind": scaler_kind,
        },
        np.asarray(transformed, dtype=np.float32),
    )


def transform_numeric_preprocessor(
    fitted,
    frame,
):
    matrix = numeric_matrix(
        frame,
        fitted["feature_columns"],
    )

    medians = fitted["medians"].reindex(
        fitted["feature_columns"]
    ).fillna(0.0)

    matrix = matrix.fillna(medians)

    transformed = (
        fitted["scaler"].transform(matrix)
        if fitted["scaler"] is not None
        else matrix.to_numpy()
    )

    return np.asarray(
        transformed,
        dtype=np.float32,
    )

In [ ]:

def calculate_binary_metrics(
    y_true,
    y_pred,
    y_score=None,
):
    y_true = np.asarray(y_true, dtype=np.int8)
    y_pred = np.asarray(y_pred, dtype=np.int8)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    metrics = {
        "n_test": int(len(y_true)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            y_pred,
        ),
        "precision": precision_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0,
        ),
        "specificity": specificity,
        "f1": f1_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0,
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "mcc": matthews_corrcoef(
            y_true,
            y_pred,
        ),
        "false_positive_rate": (
            fp / (fp + tn)
            if (fp + tn) > 0
            else np.nan
        ),
        "roc_auc": np.nan,
    }

    if (
        y_score is not None
        and len(np.unique(y_true)) == 2
    ):
        metrics["roc_auc"] = roc_auc_score(
            y_true,
            np.asarray(y_score),
        )

    return metrics


def category_recall_table(
    frame,
    y_true,
    y_pred,
):
    categories = (
        frame[LABEL_COLUMN]
        .astype(str)
        .to_numpy()
    )

    rows = []

    for category in sorted(pd.unique(categories)):
        mask = categories == category
        category_true = y_true[mask]
        category_prediction = y_pred[mask]

        rows.append({
            "test_category": category,
            "support": int(mask.sum()),
            "attack_recall": (
                recall_score(
                    category_true,
                    category_prediction,
                    pos_label=1,
                    zero_division=0,
                )
                if np.any(category_true == 1)
                else np.nan
            ),
            "benign_specificity": (
                recall_score(
                    category_true,
                    category_prediction,
                    pos_label=0,
                    zero_division=0,
                )
                if np.any(category_true == 0)
                else np.nan
            ),
        })

    return pd.DataFrame(rows)


def subtype_recall_table(
    frame,
    y_true,
    y_pred,
):
    subtypes = (
        frame[ORIGINAL_LABEL_COLUMN]
        .astype(str)
        .str.strip()
        .str.rstrip(".")
        .to_numpy()
    )

    rows = []

    for subtype in sorted(pd.unique(subtypes)):
        mask = subtypes == subtype
        subtype_true = y_true[mask]
        subtype_prediction = y_pred[mask]

        rows.append({
            "test_subtype": subtype,
            "support": int(mask.sum()),
            "attack_recall": (
                recall_score(
                    subtype_true,
                    subtype_prediction,
                    pos_label=1,
                    zero_division=0,
                )
                if np.any(subtype_true == 1)
                else np.nan
            ),
            "benign_specificity": (
                recall_score(
                    subtype_true,
                    subtype_prediction,
                    pos_label=0,
                    zero_division=0,
                )
                if np.any(subtype_true == 0)
                else np.nan
            ),
        })

    return pd.DataFrame(rows)

In [ ]:

ALZUGHABI_SELECTED_FEATURES = [
    "Dst Port",
    "Fwd Seg Size Min",
    "Init Fwd Win Byts",
    "Fwd Pkt Len Max",
    "TotLen Fwd Pkts",
    "Subflow Fwd Byts",
    "Fwd Pkt Len Mean",
    "Fwd IAT Tot",
    "Flow Duration",
    "Fwd IAT Mean",
    "Fwd IAT Max",
    "Flow IAT Max",
    "Fwd Seg Size Avg",
    "Flow IAT Min",
    "Fwd Pkts/s",
    "Flow IAT Mean",
    "Fwd IAT Min",
    "Flow Pkts/s",
    "Fwd Header Len",
    "Init Bwd Win Byts",
    "Bwd Pkts/s",
    "Pkt Len Max",
    "Tot Fwd Pkts",
    "Subflow Fwd Pkts",
]


def build_alzughabi_pso_dnn():
    learning_rate_schedule = (
        tf.keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate=0.001,
            decay_steps=10000,
            decay_rate=0.96,
            staircase=True,
        )
    )

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(24,)),
        tf.keras.layers.Dense(
            15,
            activation="softsign",
        ),
        tf.keras.layers.Dense(
            11,
            activation="relu",
        ),
        tf.keras.layers.Dense(
            2,
            activation="softmax",
        ),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate_schedule
        ),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=[
            tf.keras.metrics.CategoricalAccuracy(
                name="accuracy"
            )
        ],
    )

    return model


def fit_alzughabi_pso_dnn(
    train_frame,
    seed,
):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()

    train_frame = optional_sample(
        train_frame,
        MAX_TRAIN_ROWS,
        seed,
    )

    preprocessor, X_train = (
        fit_numeric_preprocessor(
            train_frame,
            ALZUGHABI_SELECTED_FEATURES,
            scaler_kind="standard",
            drop_constant=False,
        )
    )

    y_train = tf.keras.utils.to_categorical(
        binary_y(train_frame),
        num_classes=2,
    )

    model = build_alzughabi_pso_dnn()

    model.fit(
        X_train,
        y_train,
        validation_split=0.10,
        batch_size=BATCH_SIZES[
            "alzughabi_pso_dnn"
        ],
        epochs=EPOCHS[
            "alzughabi_pso_dnn"
        ],
        shuffle=True,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                mode="min",
                patience=8,
                restore_best_weights=True,
                min_delta=1e-4,
                verbose=1,
            )
        ],
        verbose=2,
    )

    return {
        "model": model,
        "preprocessor": preprocessor,
        "seed": seed,
    }


def predict_alzughabi_pso_dnn(
    fitted,
    test_frame,
):
    evaluated = prepare_test(
        test_frame,
        fitted["seed"],
    )

    X_test = transform_numeric_preprocessor(
        fitted["preprocessor"],
        evaluated,
    )

    probabilities = fitted["model"].predict(
        X_test,
        batch_size=BATCH_SIZES[
            "alzughabi_pso_dnn"
        ],
        verbose=0,
    )

    prediction = np.argmax(
        probabilities,
        axis=1,
    ).astype(np.int8)

    return (
        binary_y(evaluated),
        prediction,
        probabilities[:, 1],
        evaluated,
    )

In [ ]:

WANG_ALL_ZERO_FEATURES = {
    "Bwd PSH Flags",
    "Bwd URG Flags",
    "Fwd Byts/b Avg",
    "Fwd Pkts/b Avg",
    "Fwd Blk Rate Avg",
    "Bwd Byts/b Avg",
    "Bwd Pkts/b Avg",
    "Bwd Blk Rate Avg",
}

WANG_CANDIDATE_FEATURES = [
    column
    for column in CSE2018_FEATURE_COLUMNS
    if (
        column != "Dst Port"
        and column not in WANG_ALL_ZERO_FEATURES
    )
]


def fit_wang_preprocessor(train_frame):
    matrix = numeric_matrix(
        train_frame,
        WANG_CANDIDATE_FEATURES,
    )

    medians = matrix.median(axis=0).fillna(0.0)
    matrix = matrix.fillna(medians)

    active_features = [
        column
        for column in WANG_CANDIDATE_FEATURES
        if matrix[column].nunique(
            dropna=False
        ) > 1
    ]

    if len(active_features) > 69:
        active_features = active_features[:69]

    if not active_features:
        raise ValueError(
            "No Wang features remain after training-only filtering."
        )

    scaler = StandardScaler()
    standardized = scaler.fit_transform(
        matrix[active_features]
    ).astype(np.float32)

    training_minimums = standardized.min(
        axis=0,
        keepdims=True,
    )

    nonnegative_shifts = np.where(
        training_minimums < 0,
        -training_minimums,
        0.0,
    ).astype(np.float32)

    transformed = np.log1p(
        np.maximum(
            standardized + nonnegative_shifts,
            0.0,
        )
    ).astype(np.float32)

    return (
        {
            "feature_columns": active_features,
            "medians": medians,
            "scaler": scaler,
            "nonnegative_shifts": (
                nonnegative_shifts
            ),
        },
        transformed,
    )


def transform_wang_preprocessor(
    fitted,
    frame,
):
    matrix = numeric_matrix(
        frame,
        fitted["feature_columns"],
    )

    matrix = matrix.fillna(
        fitted["medians"].reindex(
            fitted["feature_columns"]
        ).fillna(0.0)
    )

    standardized = (
        fitted["scaler"]
        .transform(matrix)
        .astype(np.float32)
    )

    return np.log1p(
        np.maximum(
            standardized
            + fitted["nonnegative_shifts"],
            0.0,
        )
    ).astype(np.float32)

In [ ]:

def build_wang_cnn(n_features):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(
            shape=(n_features, 1)
        ),
        tf.keras.layers.Conv1D(
            128,
            kernel_size=2,
            padding="same",
            activation="relu",
        ),
        tf.keras.layers.MaxPooling1D(
            pool_size=2,
        ),
        tf.keras.layers.Conv1D(
            128,
            kernel_size=2,
            padding="same",
            activation="relu",
        ),
        tf.keras.layers.MaxPooling1D(
            pool_size=2,
        ),
        tf.keras.layers.Conv1D(
            256,
            kernel_size=2,
            padding="same",
            activation="relu",
        ),
        tf.keras.layers.MaxPooling1D(
            pool_size=2,
        ),
        tf.keras.layers.Conv1D(
            256,
            kernel_size=2,
            padding="same",
            activation="relu",
        ),
        tf.keras.layers.MaxPooling1D(
            pool_size=2,
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(
            1,
            activation="sigmoid",
        ),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.01
        ),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=["accuracy"],
    )

    return model


def fit_wang_cnn(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()

    train_frame = optional_sample(
        train_frame,
        MAX_TRAIN_ROWS,
        seed,
    )

    preprocessor, X_train = (
        fit_wang_preprocessor(train_frame)
    )

    model = build_wang_cnn(
        X_train.shape[1]
    )

    model.fit(
        X_train[..., np.newaxis],
        binary_y(train_frame),
        validation_split=0.10,
        batch_size=BATCH_SIZES["wang_cnn"],
        epochs=EPOCHS["wang_cnn"],
        shuffle=True,
        callbacks=standard_callbacks(
            patience=5
        ),
        verbose=2,
    )

    return {
        "model": model,
        "preprocessor": preprocessor,
        "seed": seed,
    }


def predict_wang_cnn(fitted, test_frame):
    evaluated = prepare_test(
        test_frame,
        fitted["seed"],
    )

    X_test = transform_wang_preprocessor(
        fitted["preprocessor"],
        evaluated,
    )

    probability = (
        fitted["model"]
        .predict(
            X_test[..., np.newaxis],
            batch_size=BATCH_SIZES[
                "wang_cnn"
            ],
            verbose=0,
        )
        .reshape(-1)
    )

    prediction = (
        probability >= 0.5
    ).astype(np.int8)

    return (
        binary_y(evaluated),
        prediction,
        probability,
        evaluated,
    )

In [ ]:

def build_wang_rnn(n_features):
    model_input = tf.keras.layers.Input(
        shape=(1, n_features)
    )

    x = model_input

    for _ in range(4):
        x = tf.keras.layers.SimpleRNN(
            128,
            return_sequences=True,
        )(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Dropout(0.2)(x)

    x = tf.keras.layers.SimpleRNN(
        256,
        return_sequences=False,
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    output = tf.keras.layers.Dense(
        1,
        activation="sigmoid",
    )(x)

    model = tf.keras.Model(
        model_input,
        output,
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.01
        ),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=["accuracy"],
    )

    return model


def fit_wang_rnn(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()

    train_frame = optional_sample(
        train_frame,
        MAX_TRAIN_ROWS,
        seed,
    )

    preprocessor, X_train = (
        fit_wang_preprocessor(train_frame)
    )

    model = build_wang_rnn(
        X_train.shape[1]
    )

    model.fit(
        X_train[:, np.newaxis, :],
        binary_y(train_frame),
        validation_split=0.10,
        batch_size=BATCH_SIZES["wang_rnn"],
        epochs=EPOCHS["wang_rnn"],
        shuffle=True,
        callbacks=standard_callbacks(
            patience=5
        ),
        verbose=2,
    )

    return {
        "model": model,
        "preprocessor": preprocessor,
        "seed": seed,
    }


def predict_wang_rnn(fitted, test_frame):
    evaluated = prepare_test(
        test_frame,
        fitted["seed"],
    )

    X_test = transform_wang_preprocessor(
        fitted["preprocessor"],
        evaluated,
    )

    probability = (
        fitted["model"]
        .predict(
            X_test[:, np.newaxis, :],
            batch_size=BATCH_SIZES[
                "wang_rnn"
            ],
            verbose=0,
        )
        .reshape(-1)
    )

    prediction = (
        probability >= 0.5
    ).astype(np.int8)

    return (
        binary_y(evaluated),
        prediction,
        probability,
        evaluated,
    )

In [ ]:

def build_wang_lstm(n_features):
    model_input = tf.keras.layers.Input(
        shape=(1, n_features)
    )

    x = model_input

    for _ in range(4):
        x = tf.keras.layers.LSTM(
            128,
            return_sequences=True,
        )(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Dropout(0.2)(x)

    x = tf.keras.layers.LSTM(
        256,
        return_sequences=False,
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    output = tf.keras.layers.Dense(
        1,
        activation="sigmoid",
    )(x)

    model = tf.keras.Model(
        model_input,
        output,
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.01
        ),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=["accuracy"],
    )

    return model


def fit_wang_lstm(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()

    train_frame = optional_sample(
        train_frame,
        MAX_TRAIN_ROWS,
        seed,
    )

    preprocessor, X_train = (
        fit_wang_preprocessor(train_frame)
    )

    model = build_wang_lstm(
        X_train.shape[1]
    )

    model.fit(
        X_train[:, np.newaxis, :],
        binary_y(train_frame),
        validation_split=0.10,
        batch_size=BATCH_SIZES["wang_lstm"],
        epochs=EPOCHS["wang_lstm"],
        shuffle=True,
        callbacks=standard_callbacks(
            patience=5
        ),
        verbose=2,
    )

    return {
        "model": model,
        "preprocessor": preprocessor,
        "seed": seed,
    }


def predict_wang_lstm(fitted, test_frame):
    evaluated = prepare_test(
        test_frame,
        fitted["seed"],
    )

    X_test = transform_wang_preprocessor(
        fitted["preprocessor"],
        evaluated,
    )

    probability = (
        fitted["model"]
        .predict(
            X_test[:, np.newaxis, :],
            batch_size=BATCH_SIZES[
                "wang_lstm"
            ],
            verbose=0,
        )
        .reshape(-1)
    )

    prediction = (
        probability >= 0.5
    ).astype(np.int8)

    return (
        binary_y(evaluated),
        prediction,
        probability,
        evaluated,
    )

In [ ]:

def fit_anwar_preprocessor(
    train_frame,
):
    matrix = numeric_matrix(
        train_frame,
        CSE2018_FEATURE_COLUMNS,
    )

    medians = matrix.median(axis=0).fillna(0.0)
    matrix = matrix.fillna(medians)

    retained_features = [
        column
        for column in CSE2018_FEATURE_COLUMNS
        if matrix[column].nunique(
            dropna=False
        ) > 1
    ]

    correlations = (
        matrix[retained_features]
        .corr(method="pearson")
        .abs()
    )

    upper_triangle = correlations.where(
        np.triu(
            np.ones(correlations.shape),
            k=1,
        ).astype(bool)
    )

    correlated_to_remove = [
        column
        for column in upper_triangle.columns
        if (upper_triangle[column] > 0.95).any()
    ]

    retained_features = [
        column
        for column in retained_features
        if column not in correlated_to_remove
    ]

    scaler = MinMaxScaler(
        feature_range=(0, 1)
    )

    transformed = scaler.fit_transform(
        matrix[retained_features]
    ).astype(np.float32)

    transformed = np.clip(
        transformed,
        0.0,
        1.0,
    )

    return (
        {
            "feature_columns": retained_features,
            "medians": medians,
            "scaler": scaler,
            "correlated_removed": (
                correlated_to_remove
            ),
        },
        transformed,
    )


def transform_anwar_preprocessor(
    fitted,
    frame,
):
    matrix = numeric_matrix(
        frame,
        fitted["feature_columns"],
    )

    matrix = matrix.fillna(
        fitted["medians"].reindex(
            fitted["feature_columns"]
        ).fillna(0.0)
    )

    transformed = (
        fitted["scaler"]
        .transform(matrix)
        .astype(np.float32)
    )

    return np.clip(
        transformed,
        0.0,
        1.0,
    )


def build_anwar_lightweight_1dcnn(
    n_features,
):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(
            shape=(n_features, 1)
        ),
        tf.keras.layers.Conv1D(
            filters=32,
            kernel_size=3,
            padding="same",
            activation="relu",
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv1D(
            filters=16,
            kernel_size=3,
            padding="same",
            activation="relu",
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dropout(0.30),
        tf.keras.layers.Dense(
            1,
            activation="sigmoid",
        ),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-3
        ),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=[
            tf.keras.metrics.BinaryAccuracy(
                name="accuracy"
            ),
            tf.keras.metrics.AUC(
                name="auc"
            ),
        ],
    )

    return model


def select_balanced_accuracy_threshold(
    y_true,
    probabilities,
):
    candidate_thresholds = np.round(
        np.arange(0.05, 0.951, 0.01),
        2,
    )

    scores = []

    for threshold in candidate_thresholds:
        prediction = (
            probabilities >= threshold
        ).astype(np.int8)

        scores.append(
            balanced_accuracy_score(
                y_true,
                prediction,
            )
        )

    return float(
        candidate_thresholds[
            int(np.argmax(scores))
        ]
    )


def fit_anwar_lightweight_1dcnn(
    train_frame,
    seed,
):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()

    train_frame = optional_sample(
        train_frame,
        MAX_TRAIN_ROWS,
        seed,
    )

    train_part, validation_part = train_test_split(
        train_frame,
        test_size=0.10,
        random_state=seed,
        shuffle=True,
        stratify=train_frame[BINARY_COLUMN],
    )

    preprocessor, X_train = (
        fit_anwar_preprocessor(train_part)
    )

    X_validation = (
        transform_anwar_preprocessor(
            preprocessor,
            validation_part,
        )
    )

    y_train = binary_y(train_part)
    y_validation = binary_y(
        validation_part
    )

    class_weight_values = compute_class_weight(
        class_weight="balanced",
        classes=np.array([0, 1]),
        y=y_train,
    )

    class_weight = {
        0: float(class_weight_values[0]),
        1: float(class_weight_values[1]),
    }

    training_counts = np.bincount(
        y_train,
        minlength=2,
    )

    minority_ratio = (
        training_counts.min()
        / training_counts.max()
    )

    if (
        minority_ratio < 0.25
        and len(X_train) < 1_000_000
        and training_counts.min() >= 6
    ):
        X_train, y_train = SMOTE(
            random_state=seed,
        ).fit_resample(
            X_train,
            y_train,
        )

    model = build_anwar_lightweight_1dcnn(
        X_train.shape[1]
    )

    model.fit(
        X_train[..., np.newaxis],
        y_train.astype(np.float32),
        validation_data=(
            X_validation[..., np.newaxis],
            y_validation.astype(
                np.float32
            ),
        ),
        batch_size=BATCH_SIZES[
            "anwar_lightweight_1dcnn"
        ],
        epochs=EPOCHS[
            "anwar_lightweight_1dcnn"
        ],
        shuffle=True,
        class_weight=class_weight,
        callbacks=standard_callbacks(
            monitor="val_auc",
            mode="max",
            patience=4,
        ),
        verbose=2,
    )

    validation_probability = (
        model.predict(
            X_validation[..., np.newaxis],
            batch_size=BATCH_SIZES[
                "anwar_lightweight_1dcnn"
            ],
            verbose=0,
        )
        .reshape(-1)
    )

    threshold = (
        select_balanced_accuracy_threshold(
            y_validation,
            validation_probability,
        )
    )

    return {
        "model": model,
        "preprocessor": preprocessor,
        "threshold": threshold,
        "seed": seed,
    }


def predict_anwar_lightweight_1dcnn(
    fitted,
    test_frame,
):
    evaluated = prepare_test(
        test_frame,
        fitted["seed"],
    )

    X_test = transform_anwar_preprocessor(
        fitted["preprocessor"],
        evaluated,
    )

    probability = (
        fitted["model"]
        .predict(
            X_test[..., np.newaxis],
            batch_size=BATCH_SIZES[
                "anwar_lightweight_1dcnn"
            ],
            verbose=0,
        )
        .reshape(-1)
    )

    prediction = (
        probability >= fitted["threshold"]
    ).astype(np.int8)

    return (
        binary_y(evaluated),
        prediction,
        probability,
        evaluated,
    )

In [ ]:

def fit_ensemble_feature_rf(
    train_frame,
    seed,
):
    set_all_seeds(seed)

    train_frame = optional_sample(
        train_frame,
        MAX_TRAIN_ROWS,
        seed,
    )

    preprocessor, X_train = (
        fit_numeric_preprocessor(
            train_frame,
            CSE2018_FEATURE_COLUMNS,
            scaler_kind="standard",
            drop_constant=True,
        )
    )

    y_train = binary_y(train_frame)

    feature_names = np.array(
        preprocessor["feature_columns"]
    )

    training_frame = pd.DataFrame(
        X_train,
        columns=feature_names,
    )
    training_frame[BINARY_COLUMN] = (
        y_train
    )

    target_correlations = (
        training_frame
        .corr(numeric_only=True)[
            BINARY_COLUMN
        ]
        .drop(BINARY_COLUMN)
        .abs()
    )

    correlation_features = set(
        target_correlations[
            target_correlations > 0.5
        ].index
    )

    mutual_information_k = min(
        20,
        X_train.shape[1],
    )

    mi_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=mutual_information_k,
    )

    mi_selector.fit(
        X_train,
        y_train,
    )

    mi_features = set(
        feature_names[
            mi_selector.get_support()
        ]
    )

    pca_model = PCA(
        n_components=0.95,
        svd_solver="full",
    )

    pca_model.fit(X_train)

    dominant_indices = np.unique(
        np.argmax(
            np.abs(pca_model.components_),
            axis=1,
        )
    )

    pca_features = set(
        feature_names[dominant_indices]
    )

    selected_features = sorted(
        correlation_features
        | mi_features
        | pca_features
    )

    if not selected_features:
        raise ValueError(
            "Feature-selection union is empty."
        )

    selected_indices = np.array([
        int(
            np.where(
                feature_names == feature
            )[0][0]
        )
        for feature in selected_features
    ])

    X_selected = X_train[
        :,
        selected_indices,
    ]

    model = RandomForestClassifier(
        n_estimators=10,
        max_depth=None,
        criterion="gini",
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        bootstrap=True,
        random_state=seed,
        n_jobs=-1,
    )

    model.fit(
        X_selected,
        y_train,
    )

    return {
        "model": model,
        "preprocessor": preprocessor,
        "selected_features": (
            selected_features
        ),
        "selected_indices": (
            selected_indices
        ),
        "seed": seed,
    }


def predict_ensemble_feature_rf(
    fitted,
    test_frame,
):
    evaluated = prepare_test(
        test_frame,
        fitted["seed"],
    )

    X_test = transform_numeric_preprocessor(
        fitted["preprocessor"],
        evaluated,
    )

    X_selected = X_test[
        :,
        fitted["selected_indices"],
    ]

    prediction = (
        fitted["model"]
        .predict(X_selected)
        .astype(np.int8)
    )

    probability = (
        fitted["model"]
        .predict_proba(X_selected)[:, 1]
    )

    return (
        binary_y(evaluated),
        prediction,
        probability,
        evaluated,
    )

In [ ]:

MODEL_REGISTRY = {
    "alzughabi_pso_dnn": {
        "fit": fit_alzughabi_pso_dnn,
        "predict": predict_alzughabi_pso_dnn,
    },
    "wang_cnn": {
        "fit": fit_wang_cnn,
        "predict": predict_wang_cnn,
    },
    "wang_rnn": {
        "fit": fit_wang_rnn,
        "predict": predict_wang_rnn,
    },
    "wang_lstm": {
        "fit": fit_wang_lstm,
        "predict": predict_wang_lstm,
    },
    "anwar_lightweight_1dcnn": {
        "fit": fit_anwar_lightweight_1dcnn,
        "predict": predict_anwar_lightweight_1dcnn,
    },
    "ensemble_feature_rf": {
        "fit": fit_ensemble_feature_rf,
        "predict": predict_ensemble_feature_rf,
    },
}

missing_models = [
    model
    for model in MODEL_KEYS
    if model not in MODEL_REGISTRY
]

if missing_models:
    raise KeyError(
        f"Unregistered models: {missing_models}"
    )

print("Registered models:", list(MODEL_REGISTRY))

Registered models: ['alzughabi_pso_dnn', 'wang_cnn', 'wang_rnn', 'wang_lstm', 'anwar_lightweight_1dcnn', 'ensemble_feature_rf']


In [ ]:

MODEL_TEST_RATIO_COUNTS = {
    "alzughabi_pso_dnn": {
        "normal": 4_005_868,
        "attack": 706_382,
    },
    "wang_cnn": {
        "normal": 1_898_660,
        "attack": 265_173,
    },
    "wang_rnn": {
        "normal": 1_898_660,
        "attack": 265_173,
    },
    "wang_lstm": {
        "normal": 1_898_660,
        "attack": 265_173,
    },
    "anwar_lightweight_1dcnn": {
        "normal": 2_070_696,
        "attack": 264_868,
    },
    "ensemble_feature_rf": {
        "normal": 185_932,
        "attack": 23_783,
    },
}

MODEL_TEST_RATIOS = {}

for model_key, counts in MODEL_TEST_RATIO_COUNTS.items():
    total = counts["normal"] + counts["attack"]
    MODEL_TEST_RATIOS[model_key] = {
        "normal_ratio": counts["normal"] / total,
        "attack_ratio": counts["attack"] / total,
        "original_test_normal": counts["normal"],
        "original_test_attack": counts["attack"],
        "original_test_total": total,
    }

model_ratio_table = pd.DataFrame(
    [
        {
            "model": model_key,
            "original_test_normal": values[
                "original_test_normal"
            ],
            "original_test_attack": values[
                "original_test_attack"
            ],
            "target_normal_pct": (
                100 * values["normal_ratio"]
            ),
            "target_attack_pct": (
                100 * values["attack_ratio"]
            ),
        }
        for model_key, values in (
            MODEL_TEST_RATIOS.items()
        )
    ]
)

display(model_ratio_table)


,model,original_test_normal,original_test_attack,target_normal_pct,target_attack_pct
0,alzughabi_pso_dnn,4005868,706382,85.009666,14.990334
1,wang_cnn,1898660,265173,87.745219,12.254781
2,wang_rnn,1898660,265173,87.745219,12.254781
3,wang_lstm,1898660,265173,87.745219,12.254781
4,anwar_lightweight_1dcnn,2070696,264868,88.659356,11.340644
5,ensemble_feature_rf,185932,23783,88.659371,11.340629


In [ ]:

NATIVE_TEST_DATES = {
    "2018-02-28",
    "2018-03-01",
    "2018-03-02",
}


def split_source_frame(
    frame,
    test_size=0.20,
    seed=PARTITION_SEED,
):
    train_parts = []
    test_parts = []

    for binary_class, class_frame in frame.groupby(
        BINARY_COLUMN,
        sort=True,
    ):
        class_frame = class_frame.sample(
            frac=1,
            random_state=seed + int(binary_class),
        )

        if len(class_frame) == 1:
            train_parts.append(class_frame)
            continue

        n_test = max(
            1,
            int(round(len(class_frame) * test_size)),
        )
        n_test = min(
            n_test,
            len(class_frame) - 1,
        )

        test_parts.append(class_frame.iloc[:n_test])
        train_parts.append(class_frame.iloc[n_test:])

    train_frame = (
        pd.concat(train_parts, ignore_index=True)
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )

    test_frame = (
        pd.concat(test_parts, ignore_index=True)
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )

    return train_frame, test_frame


def balance_source_binary_training(
    frame,
    seed,
):
    counts = (
        frame.groupby(
            [SOURCE_COLUMN, BINARY_COLUMN]
        )
        .size()
        .unstack(fill_value=0)
    )

    if not {0, 1}.issubset(counts.columns):
        raise ValueError(
            "Every included source must contain both classes."
        )

    quota = int(counts[[0, 1]].min().min())

    if quota < 1:
        raise ValueError(
            f"Invalid source/class quota: {quota}"
        )

    parts = []

    for source in sorted(
        frame[SOURCE_COLUMN].unique()
    ):
        for binary_class in [0, 1]:
            cell = frame[
                frame[SOURCE_COLUMN].eq(source)
                & frame[BINARY_COLUMN].eq(
                    binary_class
                )
            ]

            parts.append(
                cell.sample(
                    n=quota,
                    replace=False,
                    random_state=seed,
                )
            )

    balanced = (
        pd.concat(parts, ignore_index=True)
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )

    return balanced, quota


native_train = native_df[
    ~native_df["capture_date"].isin(
        NATIVE_TEST_DATES
    )
].reset_index(drop=True)

native_test = native_df[
    native_df["capture_date"].isin(
        NATIVE_TEST_DATES
    )
].reset_index(drop=True)

external_train_frames = {}
external_test_frames = {}

for source, source_frame in df_master.groupby(
    SOURCE_COLUMN,
    sort=True,
):
    train_part, test_part = split_source_frame(
        source_frame,
        test_size=0.20,
        seed=PARTITION_SEED,
    )
    external_train_frames[source] = train_part
    external_test_frames[source] = test_part

mixed_train_reference = pd.concat(
    [
        native_train,
        *external_train_frames.values(),
    ],
    ignore_index=True,
)

mixed_test_sets = {
    "CSE-CIC-IDS2018-native": native_test,
    **external_test_frames,
}

partition_audit = pd.DataFrame(
    [
        {
            "test_source": source,
            "test_rows": len(frame),
            "test_normal": int(
                (frame[BINARY_COLUMN] == 0).sum()
            ),
            "test_attack": int(
                (frame[BINARY_COLUMN] == 1).sum()
            ),
            "natural_normal_pct": (
                100
                * (frame[BINARY_COLUMN] == 0).mean()
            ),
            "natural_attack_pct": (
                100
                * (frame[BINARY_COLUMN] == 1).mean()
            ),
        }
        for source, frame in mixed_test_sets.items()
    ]
)

display(partition_audit)


,test_source,test_rows,test_normal,test_attack,natural_normal_pct,natural_attack_pct
0,CSE-CIC-IDS2018-native,1992804,1544621,448183,77.509931,22.490069
1,CIC-IDS2017,246625,211070,35555,85.583376,14.416624
2,CSE-CIC-IDS2018,332335,308055,24280,92.694119,7.305881
3,DARPA,487830,452808,35022,92.820860,7.179140
4,UNSW-NB15,62727,59819,2908,95.364038,4.635962


In [ ]:

def choose_binary_sample_counts(
    normal_available,
    attack_available,
    target_normal_ratio,
):

    normal_available = int(normal_available)
    attack_available = int(attack_available)
    target_normal_ratio = float(
        target_normal_ratio
    )
    target_attack_ratio = (
        1.0 - target_normal_ratio
    )

    if normal_available < 1 or attack_available < 1:
        raise ValueError(
            "Ratio matching requires both normal and attack rows."
        )

    if not 0.0 < target_normal_ratio < 1.0:
        raise ValueError(
            "Target normal ratio must be strictly between 0 and 1."
        )

    candidates = set()

    def add_candidate(normal_count, attack_count):
        normal_count = int(normal_count)
        attack_count = int(attack_count)

        if (
            normal_count >= 1
            and attack_count >= 1
            and normal_count <= normal_available
            and attack_count <= attack_available
        ):
            candidates.add(
                (normal_count, attack_count)
            )

    ideal_attack_from_all_normal = (
        normal_available
        * target_attack_ratio
        / target_normal_ratio
    )

    for attack_count in {
        math.floor(ideal_attack_from_all_normal),
        round(ideal_attack_from_all_normal),
        math.ceil(ideal_attack_from_all_normal),
    }:
        add_candidate(
            normal_available,
            attack_count,
        )

    ideal_normal_from_all_attack = (
        attack_available
        * target_normal_ratio
        / target_attack_ratio
    )

    for normal_count in {
        math.floor(ideal_normal_from_all_attack),
        round(ideal_normal_from_all_attack),
        math.ceil(ideal_normal_from_all_attack),
    }:
        add_candidate(
            normal_count,
            attack_available,
        )

    if not candidates:
        raise ValueError(
            "No feasible two-class sample could be allocated."
        )

    def candidate_key(pair):
        normal_count, attack_count = pair
        total = normal_count + attack_count
        achieved = normal_count / total
        error = abs(
            achieved - target_normal_ratio
        )

        return (-total, error)

    return min(candidates, key=candidate_key)


def proportional_group_sample(
    frame,
    sample_size,
    group_column,
    seed,
):
    """Sample a requested total while approximately preserving group shares."""
    sample_size = int(sample_size)

    if sample_size < 0 or sample_size > len(frame):
        raise ValueError(
            "Requested grouped sample size is infeasible."
        )

    if sample_size == len(frame):
        return (
            frame.sample(
                frac=1,
                random_state=seed,
            )
            .reset_index(drop=True)
        )

    if sample_size == 0:
        return frame.iloc[0:0].copy()

    working = frame.copy()
    working["_ratio_group"] = (
        working[group_column]
        .astype("string")
        .fillna("missing")
    )

    counts = (
        working["_ratio_group"]
        .value_counts(sort=False)
        .sort_index()
    )

    ideal = (
        counts.astype(float)
        / counts.sum()
        * sample_size
    )

    allocation = np.floor(
        ideal
    ).astype(int)

    allocation = pd.Series(
        np.minimum(
            allocation.to_numpy(),
            counts.to_numpy(),
        ),
        index=counts.index,
        dtype=int,
    )

    remaining = (
        sample_size - int(allocation.sum())
    )

    fractional = (
        ideal - np.floor(ideal)
    )

    priority = sorted(
        counts.index,
        key=lambda group: (
            -float(fractional.loc[group]),
            str(group),
        ),
    )

    while remaining > 0:
        made_progress = False

        for group in priority:
            if allocation.loc[group] < counts.loc[group]:
                allocation.loc[group] += 1
                remaining -= 1
                made_progress = True

                if remaining == 0:
                    break

        if not made_progress:
            raise RuntimeError(
                "Could not distribute grouped sample remainder."
            )

    parts = []

    for offset, group in enumerate(
        allocation.index
    ):
        take = int(allocation.loc[group])

        if take == 0:
            continue

        group_frame = working[
            working["_ratio_group"].eq(group)
        ].drop(columns=["_ratio_group"])

        parts.append(
            group_frame.sample(
                n=take,
                replace=False,
                random_state=seed + offset + 1,
            )
        )

    return (
        pd.concat(parts, ignore_index=True)
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )


def match_model_test_ratio(
    frame,
    model_key,
    seed,
):
    ratio = MODEL_TEST_RATIOS[model_key]
    target_normal_ratio = ratio[
        "normal_ratio"
    ]

    normal_frame = frame[
        frame[BINARY_COLUMN].eq(0)
    ]
    attack_frame = frame[
        frame[BINARY_COLUMN].eq(1)
    ]

    normal_selected, attack_selected = (
        choose_binary_sample_counts(
            len(normal_frame),
            len(attack_frame),
            target_normal_ratio,
        )
    )

    sampled_normal = normal_frame.sample(
        n=normal_selected,
        replace=False,
        random_state=seed,
    )

    sampled_attack = proportional_group_sample(
        attack_frame,
        attack_selected,
        LABEL_COLUMN,
        seed + 10_000,
    )

    matched = (
        pd.concat(
            [sampled_normal, sampled_attack],
            ignore_index=True,
        )
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )

    actual_normal = int(
        (matched[BINARY_COLUMN] == 0).sum()
    )
    actual_attack = int(
        (matched[BINARY_COLUMN] == 1).sum()
    )
    actual_total = len(matched)

    audit = {
        "model": model_key,
        "available_rows": len(frame),
        "available_normal": len(normal_frame),
        "available_attack": len(attack_frame),
        "selected_rows": actual_total,
        "selected_normal": actual_normal,
        "selected_attack": actual_attack,
        "target_normal_pct": (
            100 * target_normal_ratio
        ),
        "target_attack_pct": (
            100 * (1.0 - target_normal_ratio)
        ),
        "actual_normal_pct": (
            100 * actual_normal / actual_total
        ),
        "actual_attack_pct": (
            100 * actual_attack / actual_total
        ),
    }

    return matched, audit


In [ ]:

def merge_and_save(
    path,
    new_rows,
    key_columns,
):
    path = Path(path)

    if path.exists():
        existing = pd.read_csv(path)
        combined = pd.concat(
            [existing, new_rows],
            ignore_index=True,
        )
    else:
        combined = new_rows.copy()

    combined = (
        combined
        .drop_duplicates(
            subset=key_columns,
            keep="last",
        )
        .reset_index(drop=True)
    )

    combined.to_csv(path, index=False)
    return combined


In [ ]:

MIXED_EXPERIMENT = (
    "exp4_mixed_traffic_model_specific_test_ratios"
)


def evaluate_model_specific_ratio_experiment(
    model_key,
    seed,
):
    adapter = MODEL_REGISTRY[model_key]

    train_frame, source_class_quota = (
        balance_source_binary_training(
            mixed_train_reference,
            seed,
        )
    )

    target = MODEL_TEST_RATIOS[model_key]

    print(
        f"\n{MIXED_EXPERIMENT} | "
        f"{model_key} | seed={seed}",
        flush=True,
    )
    print(
        "  Target test prevalence: "
        f"{100 * target['normal_ratio']:.6f}% normal, "
        f"{100 * target['attack_ratio']:.6f}% attack",
        flush=True,
    )
    print(
        f"  Balanced training rows: "
        f"{len(train_frame):,}; "
        f"quota={source_class_quota:,}",
        flush=True,
    )

    fit_started = time.perf_counter()
    fitted = adapter["fit"](
        train_frame,
        seed,
    )
    fit_seconds = (
        time.perf_counter() - fit_started
    )

    overall_rows = []
    category_frames = []
    subtype_frames = []
    ratio_rows = []

    pooled_true = []
    pooled_prediction = []
    pooled_score = []
    pooled_evaluated = []

    for source_offset, (
        test_source,
        raw_test_frame,
    ) in enumerate(mixed_test_sets.items()):
        matched_test_frame, ratio_audit = (
            match_model_test_ratio(
                raw_test_frame,
                model_key,
                seed=(
                    seed
                    + 100_000
                    + 1_000 * source_offset
                ),
            )
        )

        ratio_audit.update({
            "experiment": MIXED_EXPERIMENT,
            "seed": seed,
            "test_source": test_source,
        })
        ratio_rows.append(ratio_audit)

        print(
            f"  Testing {test_source}: "
            f"{len(raw_test_frame):,} available -> "
            f"{len(matched_test_frame):,} matched; "
            f"normal={ratio_audit['actual_normal_pct']:.6f}%",
            flush=True,
        )

        prediction_started = time.perf_counter()

        (
            y_true,
            y_pred,
            y_score,
            evaluated,
        ) = adapter["predict"](
            fitted,
            matched_test_frame,
        )

        prediction_seconds = (
            time.perf_counter()
            - prediction_started
        )

        metrics = calculate_binary_metrics(
            y_true,
            y_pred,
            y_score,
        )

        evaluated_normal = int(
            np.sum(y_true == 0)
        )
        evaluated_attack = int(
            np.sum(y_true == 1)
        )
        evaluated_total = len(y_true)

        overall_rows.append({
            "experiment": MIXED_EXPERIMENT,
            "training_variant": "balanced",
            "model": model_key,
            "seed": seed,
            "test_source": test_source,
            "source_class_quota": (
                source_class_quota
            ),
            "n_train_used": len(train_frame),
            "n_test_raw": len(raw_test_frame),
            "n_test_ratio_matched": (
                len(matched_test_frame)
            ),
            "n_test_evaluated": evaluated_total,
            "target_normal_pct": (
                100 * target["normal_ratio"]
            ),
            "target_attack_pct": (
                100 * target["attack_ratio"]
            ),
            "actual_normal_pct": (
                100
                * evaluated_normal
                / evaluated_total
            ),
            "actual_attack_pct": (
                100
                * evaluated_attack
                / evaluated_total
            ),
            "fit_seconds": fit_seconds,
            "prediction_seconds": (
                prediction_seconds
            ),
            **metrics,
        })

        category_table = category_recall_table(
            evaluated,
            y_true,
            y_pred,
        )
        category_table["experiment"] = (
            MIXED_EXPERIMENT
        )
        category_table["training_variant"] = (
            "balanced"
        )
        category_table["model"] = model_key
        category_table["seed"] = seed
        category_table["test_source"] = (
            test_source
        )
        category_frames.append(category_table)

        subtype_table = subtype_recall_table(
            evaluated,
            y_true,
            y_pred,
        )
        subtype_table["experiment"] = (
            MIXED_EXPERIMENT
        )
        subtype_table["training_variant"] = (
            "balanced"
        )
        subtype_table["model"] = model_key
        subtype_table["seed"] = seed
        subtype_table["test_source"] = (
            test_source
        )
        subtype_frames.append(subtype_table)

        pooled_true.append(y_true)
        pooled_prediction.append(y_pred)
        pooled_score.append(y_score)
        pooled_evaluated.append(evaluated)

    pooled_true_array = np.concatenate(
        pooled_true
    )
    pooled_prediction_array = np.concatenate(
        pooled_prediction
    )
    pooled_score_array = np.concatenate(
        pooled_score
    )

    pooled_metrics = calculate_binary_metrics(
        pooled_true_array,
        pooled_prediction_array,
        pooled_score_array,
    )

    pooled_normal = int(
        np.sum(pooled_true_array == 0)
    )
    pooled_attack = int(
        np.sum(pooled_true_array == 1)
    )
    pooled_total = len(pooled_true_array)

    overall_rows.append({
        "experiment": MIXED_EXPERIMENT,
        "training_variant": "balanced",
        "model": model_key,
        "seed": seed,
        "test_source": "POOLED_ALL_ROWS",
        "source_class_quota": (
            source_class_quota
        ),
        "n_train_used": len(train_frame),
        "n_test_raw": int(
            sum(
                len(frame)
                for frame in mixed_test_sets.values()
            )
        ),
        "n_test_ratio_matched": pooled_total,
        "n_test_evaluated": pooled_total,
        "target_normal_pct": (
            100 * target["normal_ratio"]
        ),
        "target_attack_pct": (
            100 * target["attack_ratio"]
        ),
        "actual_normal_pct": (
            100 * pooled_normal / pooled_total
        ),
        "actual_attack_pct": (
            100 * pooled_attack / pooled_total
        ),
        "fit_seconds": fit_seconds,
        "prediction_seconds": np.nan,
        **pooled_metrics,
    })

    ratio_rows.append({
        "experiment": MIXED_EXPERIMENT,
        "model": model_key,
        "seed": seed,
        "test_source": "POOLED_ALL_ROWS",
        "available_rows": int(
            sum(
                len(frame)
                for frame in mixed_test_sets.values()
            )
        ),
        "available_normal": int(
            sum(
                (frame[BINARY_COLUMN] == 0).sum()
                for frame in mixed_test_sets.values()
            )
        ),
        "available_attack": int(
            sum(
                (frame[BINARY_COLUMN] == 1).sum()
                for frame in mixed_test_sets.values()
            )
        ),
        "selected_rows": pooled_total,
        "selected_normal": pooled_normal,
        "selected_attack": pooled_attack,
        "target_normal_pct": (
            100 * target["normal_ratio"]
        ),
        "target_attack_pct": (
            100 * target["attack_ratio"]
        ),
        "actual_normal_pct": (
            100 * pooled_normal / pooled_total
        ),
        "actual_attack_pct": (
            100 * pooled_attack / pooled_total
        ),
    })

    del fitted
    gc.collect()
    tf.keras.backend.clear_session()

    return (
        pd.DataFrame(overall_rows),
        pd.concat(
            category_frames,
            ignore_index=True,
        ),
        pd.concat(
            subtype_frames,
            ignore_index=True,
        ),
        pd.DataFrame(ratio_rows),
    )


In [ ]:

def run_model_specific_ratio_experiment():
    overall_path = (
        RESULTS_DIR
        / f"{MIXED_EXPERIMENT}_all_runs_partial.csv"
    )
    category_path = (
        RESULTS_DIR
        / f"{MIXED_EXPERIMENT}_category_runs_partial.csv"
    )
    subtype_path = (
        RESULTS_DIR
        / f"{MIXED_EXPERIMENT}_subtype_runs_partial.csv"
    )
    ratio_path = (
        RESULTS_DIR
        / f"{MIXED_EXPERIMENT}_ratio_manifest_partial.csv"
    )

    completed = (
        pd.read_csv(overall_path)
        if overall_path.exists()
        else pd.DataFrame()
    )

    overall_accumulated = completed.copy()
    category_accumulated = (
        pd.read_csv(category_path)
        if category_path.exists()
        else pd.DataFrame()
    )
    subtype_accumulated = (
        pd.read_csv(subtype_path)
        if subtype_path.exists()
        else pd.DataFrame()
    )
    ratio_accumulated = (
        pd.read_csv(ratio_path)
        if ratio_path.exists()
        else pd.DataFrame()
    )

    for model_key in MODEL_KEYS:
        for seed in SEEDS:
            if not completed.empty:
                complete_mask = (
                    completed["experiment"].eq(
                        MIXED_EXPERIMENT
                    )
                    & completed["model"].eq(
                        model_key
                    )
                    & completed["seed"].eq(seed)
                    & completed["test_source"].eq(
                        "POOLED_ALL_ROWS"
                    )
                )
                is_complete = bool(
                    complete_mask.any()
                )
            else:
                is_complete = False

            if is_complete:
                print(
                    f"SKIP completed: "
                    f"{model_key}, seed={seed}",
                    flush=True,
                )
                continue

            (
                overall,
                categories,
                subtypes,
                ratios,
            ) = (
                evaluate_model_specific_ratio_experiment(
                    model_key,
                    seed,
                )
            )

            category_accumulated = merge_and_save(
                category_path,
                categories,
                [
                    "experiment",
                    "training_variant",
                    "model",
                    "seed",
                    "test_source",
                    "test_category",
                ],
            )

            subtype_accumulated = merge_and_save(
                subtype_path,
                subtypes,
                [
                    "experiment",
                    "training_variant",
                    "model",
                    "seed",
                    "test_source",
                    "test_subtype",
                ],
            )

            ratio_accumulated = merge_and_save(
                ratio_path,
                ratios,
                [
                    "experiment",
                    "model",
                    "seed",
                    "test_source",
                ],
            )

            overall_accumulated = merge_and_save(
                overall_path,
                overall,
                [
                    "experiment",
                    "training_variant",
                    "model",
                    "seed",
                    "test_source",
                ],
            )

            completed = overall_accumulated

    return (
        overall_accumulated,
        category_accumulated,
        subtype_accumulated,
        ratio_accumulated,
    )


In [31]:
(
    exp4_overall,
    exp4_categories,
    exp4_subtypes,
    exp4_ratio_manifest,
) = run_model_specific_ratio_experiment()



exp4_mixed_traffic_model_specific_test_ratios | alzughabi_pso_dnn | seed=11
  Target test prevalence: 85.009666% normal, 14.990334% attack
  Balanced training rows: 116,310; quota=11,631
Epoch 1/100
52/52 - 10s - 193ms/step - accuracy: 0.7619 - loss: 0.6020 - val_accuracy: 0.8103 - val_loss: 0.5480
Epoch 2/100
52/52 - 0s - 4ms/step - accuracy: 0.8178 - loss: 0.5082 - val_accuracy: 0.8194 - val_loss: 0.4773
Epoch 3/100
52/52 - 0s - 4ms/step - accuracy: 0.8285 - loss: 0.4520 - val_accuracy: 0.8223 - val_loss: 0.4439
Epoch 4/100
52/52 - 0s - 4ms/step - accuracy: 0.8295 - loss: 0.4262 - val_accuracy: 0.8231 - val_loss: 0.4272
Epoch 5/100
52/52 - 0s - 4ms/step - accuracy: 0.8370 - loss: 0.4098 - val_accuracy: 0.8329 - val_loss: 0.4130
Epoch 6/100
52/52 - 0s - 4ms/step - accuracy: 0.8444 - loss: 0.3949 - val_accuracy: 0.8364 - val_loss: 0.3984
Epoch 7/100
52/52 - 0s - 4ms/step - accuracy: 0.8483 - loss: 0.3805 - val_accuracy: 0.8412 - val_loss: 0.3851
Epoch 8/100
52/52 - 0s - 4ms/step - acc

In [ ]:

REPORT_METRICS = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "macro_f1",
    "mcc",
    "false_positive_rate",
    "roc_auc",
]

exp4_per_source_summary = (
    exp4_overall
    .groupby(
        [
            "experiment",
            "training_variant",
            "model",
            "test_source",
        ],
        dropna=False,
    )[REPORT_METRICS]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
)

exp4_source_macro_by_seed = (
    exp4_overall[
        exp4_overall["test_source"].ne(
            "POOLED_ALL_ROWS"
        )
    ]
    .groupby(
        [
            "experiment",
            "training_variant",
            "model",
            "seed",
        ],
        dropna=False,
    )
    .agg(
        source_macro_f1=("f1", "mean"),
        source_macro_balanced_accuracy=(
            "balanced_accuracy",
            "mean",
        ),
        source_macro_recall=("recall", "mean"),
        source_macro_specificity=(
            "specificity",
            "mean",
        ),
    )
    .reset_index()
)

exp4_source_macro_summary = (
    exp4_source_macro_by_seed
    .groupby(
        [
            "experiment",
            "training_variant",
            "model",
        ],
        dropna=False,
    )[
        [
            "source_macro_f1",
            "source_macro_balanced_accuracy",
            "source_macro_recall",
            "source_macro_specificity",
        ]
    ]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
)

exp4_category_summary = (
    exp4_categories
    .groupby(
        [
            "experiment",
            "training_variant",
            "model",
            "test_source",
            "test_category",
        ],
        dropna=False,
    )
    .agg(
        support=("support", "mean"),
        attack_recall_mean=(
            "attack_recall",
            "mean",
        ),
        attack_recall_std=(
            "attack_recall",
            "std",
        ),
        benign_specificity_mean=(
            "benign_specificity",
            "mean",
        ),
        benign_specificity_std=(
            "benign_specificity",
            "std",
        ),
    )
    .reset_index()
)

exp4_ratio_summary = (
    exp4_ratio_manifest
    .groupby(
        [
            "experiment",
            "model",
            "test_source",
        ],
        dropna=False,
    )
    .agg(
        target_normal_pct=(
            "target_normal_pct",
            "first",
        ),
        target_attack_pct=(
            "target_attack_pct",
            "first",
        ),
        selected_rows_mean=(
            "selected_rows",
            "mean",
        ),
        actual_normal_pct_mean=(
            "actual_normal_pct",
            "mean",
        ),
        actual_normal_pct_min=(
            "actual_normal_pct",
            "min",
        ),
        actual_normal_pct_max=(
            "actual_normal_pct",
            "max",
        ),
        actual_attack_pct_mean=(
            "actual_attack_pct",
            "mean",
        ),
    )
    .reset_index()
)

exports = {
    f"{MIXED_EXPERIMENT}_all_runs.csv": (
        exp4_overall
    ),
    f"{MIXED_EXPERIMENT}_category_runs.csv": (
        exp4_categories
    ),
    f"{MIXED_EXPERIMENT}_subtype_runs.csv": (
        exp4_subtypes
    ),
    f"{MIXED_EXPERIMENT}_ratio_manifest.csv": (
        exp4_ratio_manifest
    ),
    f"{MIXED_EXPERIMENT}_per_source_summary.csv": (
        exp4_per_source_summary
    ),
    f"{MIXED_EXPERIMENT}_source_macro_by_seed.csv": (
        exp4_source_macro_by_seed
    ),
    f"{MIXED_EXPERIMENT}_source_macro_summary.csv": (
        exp4_source_macro_summary
    ),
    f"{MIXED_EXPERIMENT}_category_summary.csv": (
        exp4_category_summary
    ),
    f"{MIXED_EXPERIMENT}_ratio_summary.csv": (
        exp4_ratio_summary
    ),
}

for filename, frame in exports.items():
    frame.to_csv(
        RESULTS_DIR / filename,
        index=False,
    )

display(exp4_per_source_summary)
display(exp4_source_macro_summary)
display(exp4_ratio_summary)


experiment training_variant  \
                                                                     
0   exp4_mixed_traffic_model_specific_test_ratios         balanced   
1   exp4_mixed_traffic_model_specific_test_ratios         balanced   
2   exp4_mixed_traffic_model_specific_test_ratios         balanced   
3   exp4_mixed_traffic_model_specific_test_ratios         balanced   
4   exp4_mixed_traffic_model_specific_test_ratios         balanced   
5   exp4_mixed_traffic_model_specific_test_ratios         balanced   
6   exp4_mixed_traffic_model_specific_test_ratios         balanced   
7   exp4_mixed_traffic_model_specific_test_ratios         balanced   
8   exp4_mixed_traffic_model_specific_test_ratios         balanced   
9   exp4_mixed_traffic_model_specific_test_ratios         balanced   
10  exp4_mixed_traffic_model_specific_test_ratios         balanced   
11  exp4_mixed_traffic_model_specific_test_ratios         balanced   
12  exp4_mixed_traffic_model_specific_test_ratios         balanced   
13  exp4_mixed_traffic_model_specific_test_ratios         balanced   
14  exp4_mixed_traffic_model_specific_test_ratios         balanced   
15  exp4_mixed_traffic_model_specific_test_ratios         balanced   
16  exp4_mixed_traffic_model_specific_test_ratios         balanced   
17  exp4_mixed_traffic_model_specific_test_ratios         balanced   
18  exp4_mixed_traffic_model_specific_test_ratios         balanced   
19  exp4_mixed_traffic_model_specific_test_ratios         balanced   
20  exp4_mixed_traffic_model_specific_test_ratios         balanced   
21  exp4_mixed_traffic_model_specific_test_ratios         balanced   
22  exp4_mixed_traffic_model_specific_test_ratios         balanced   
23  exp4_mixed_traffic_model_specific_test_ratios         balanced   
24  exp4_mixed_traffic_model_specific_test_ratios         balanced   
25  exp4_mixed_traffic_model_specific_test_ratios         balanced   
26  exp4_mixed_traffic_model_specific_test_ratios         balanced   
27  exp4_mixed_traffic_model_specific_test_ratios         balanced   
28  exp4_mixed_traffic_model_specific_test_ratios         balanced   
29  exp4_mixed_traffic_model_specific_test_ratios         balanced   
30  exp4_mixed_traffic_model_specific_test_ratios         balanced   
31  exp4_mixed_traffic_model_specific_test_ratios         balanced   
32  exp4_mixed_traffic_model_specific_test_ratios         balanced   
33  exp4_mixed_traffic_model_specific_test_ratios         balanced   
34  exp4_mixed_traffic_model_specific_test_ratios         balanced   
35  exp4_mixed_traffic_model_specific_test_ratios         balanced   

                      model             test_source  accuracy            \
                                                         mean       std   
0         alzughabi_pso_dnn             CIC-IDS2017  0.965410  0.002866   
1         alzughabi_pso_dnn         CSE-CIC-IDS2018  0.863485  0.004237   
2         alzughabi_pso_dnn  CSE-CIC-IDS2018-native  0.843935  0.004588   
3         alzughabi_pso_dnn                   DARPA  0.971598  0.001388   
4         alzughabi_pso_dnn         POOLED_ALL_ROWS  0.869895  0.003685   
5         alzughabi_pso_dnn               UNSW-NB15  0.962258  0.001065   
6   anwar_lightweight_1dcnn             CIC-IDS2017  0.940264  0.021732   
7   anwar_lightweight_1dcnn         CSE-CIC-IDS2018  0.835506  0.026446   
8   anwar_lightweight_1dcnn  CSE-CIC-IDS2018-native  0.833801  0.020362   
9   anwar_lightweight_1dcnn                   DARPA  0.960770  0.006134   
10  anwar_lightweight_1dcnn         POOLED_ALL_ROWS  0.860744  0.015570   
11  anwar_lightweight_1dcnn               UNSW-NB15  0.959100  0.001524   
12      ensemble_feature_rf             CIC-IDS2017  0.999371  0.000125   
13      ensemble_feature_rf         CSE-CIC-IDS2018  0.846529  0.002383   
14      ensemble_feature_rf  CSE-CIC-IDS2018-native  0.766186  0.004767   
15      ensemble_feature_rf                   DARPA  0.995335  0.000590   
16      ensemble_feature_rf 

experiment training_variant  \
                                                                    
0  exp4_mixed_traffic_model_specific_test_ratios         balanced   
1  exp4_mixed_traffic_model_specific_test_ratios         balanced   
2  exp4_mixed_traffic_model_specific_test_ratios         balanced   
3  exp4_mixed_traffic_model_specific_test_ratios         balanced   
4  exp4_mixed_traffic_model_specific_test_ratios         balanced   
5  exp4_mixed_traffic_model_specific_test_ratios         balanced   

                     model source_macro_f1                                \
                                      mean       std       min       max   
0        alzughabi_pso_dnn        0.730951  0.003077  0.727302  0.734673   
1  anwar_lightweight_1dcnn        0.639633  0.031078  0.602147  0.679673   
2      ensemble_feature_rf        0.692667  0.007923  0.685663  0.703351   
3                 wang_cnn        0.744545  0.002546  0.740977  0.748168   
4                wang_lstm        0.716928  0.038070  0.687389  0.760346   
5                 wang_rnn        0.687510  0.002178  0.683946  0.689309   

  source_macro_balanced_accuracy                                \
                            mean       std       min       max   
0                       0.858805  0.000706  0.857845  0.859576   
1                       0.833042  0.022234  0.801647  0.864472   
2                       0.843826  0.005565  0.838561  0.851289   
3                       0.868409  0.000769  0.867639  0.869498   
4                       0.853194  0.017181  0.840295  0.872588   
5                       0.839767  0.000791  0.838491  0.840533   

  source_macro_recall                               source_macro_specificity  \
                 mean       std       min       max                     mean   
0            0.769498  0.001872  0.768437  0.772802                 0.948112   
1            0.738826  0.055257  0.654017  0.806736                 0.927258   
2            0.751899  0.012376  0.739760  0.768541                 0.935753   
3            0.773569  0.002784  0.770794  0.777550                 0.963249   
4            0.735861  0.035353  0.706600  0.775002                 0.970527   
5            0.707881  0.002858  0.703948  0.710811                 0.971652   

                                 
        std       min       max  
0  0.002577  0.944111  0.950363  
1  0.016214  0.912720  0.949277  
2  0.001536  0.934037  0.937361  
3  0.002084  0.959706  0.964653  
4  0.002234  0.968419  0.973990  
5  0.002253  0.970013  0.975559

,experiment,model,test_source,target_normal_pct,target_attack_pct,selected_rows_mean,actual_normal_pct_mean,actual_normal_pct_min,actual_normal_pct_max,actual_attack_pct_mean
0,exp4_mixed_traffic_model_specific_test_ratios,alzughabi_pso_dnn,CIC-IDS2017,85.009666,14.990334,237187.0,85.009718,85.009718,85.009718,14.990282
1,exp4_mixed_traffic_model_specific_test_ratios,alzughabi_pso_dnn,CSE-CIC-IDS2018,85.009666,14.990334,161972.0,85.009755,85.009755,85.009755,14.990245
2,exp4_mixed_traffic_model_specific_test_ratios,alzughabi_pso_dnn,CSE-CIC-IDS2018-native,85.009666,14.990334,1816995.0,85.009645,85.009645,85.009645,14.990355
3,exp4_mixed_traffic_model_specific_test_ratios,alzughabi_pso_dnn,DARPA,85.009666,14.990334,233631.0,85.009695,85.009695,85.009695,14.990305
4,exp4_mixed_traffic_model_specific_test_ratios,alzughabi_pso_dnn,POOLED_ALL_ROWS,85.009666,14.990334,2469185.0,85.009669,85.009669,85.009669,14.990331
5,exp4_mixed_traffic_model_specific_test_ratios,alzughabi_pso_dnn,UNSW-NB15,85.009666,14.990334,19400.0,85.010309,85.010309,85.010309,14.989691
6,exp4_mixed_traffic_model_specific_test_ratios,anwar_lightweight_1dcnn,CIC-IDS2017,88.659356,11.340644,238069.0,88.659170,88.659170,88.659170,11.340830
7,exp4_mixed_traffic_model_specific_test_ratios,anwar_lightweight_1dcnn,CSE-CIC-IDS2018,88.659356,11.340644,214098.0,88.659399,88.659399,88.659399,11.340601
8,exp4_mixed_traffic_model_specific_test_ratios,anwar_lightweight_1dcnn,CSE-CIC-IDS2018-native,88.659356,11.340644,1742198.0,88.659326,88.659326,88.659326,11.340674
9,exp4_mixed_traffic_model_specific_test_ratios,anwar_lightweight_1dcnn,DARPA,88.659356,11.340644,308819.0,88.659377,88.659377,88.659377,11.340623
